# Review data in app.sqlite

This notebook loads the local SQLite database and shows users, reviews, entities, and review-entity links.

In [25]:
import pandas as pd
import sqlite3
from pathlib import Path

db_path = Path('../entity-driven/data/app.sqlite')
if not db_path.exists():
    raise FileNotFoundError(f'Database not found at {db_path.resolve()}')

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

def show_old(query, params=None):
    params = params or ()
    cur = conn.execute(query, params)
    rows = cur.fetchall()
    return [dict(row) for row in rows]


def show(query, params=None):
    params = params or ()
    cur = conn.execute(query, params)
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=rows[0].keys() if rows else [])


In [26]:
cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

tables = [row[0] for row in cursor.fetchall()]
print(tables)

['edges', 'entity', 'nodes', 'review', 'review_entity', 'sqlite_sequence', 'user']


In [29]:
show('SELECT * FROM nodes')

,id,name,type
0,1,BHC 쏘마치,default
1,2,처가집 양념통닭 K마라치킨,default
2,3,멕시카나,default
3,4,배,default
4,5,비비큐 맵소,default
5,6,노랑통닭,default
6,9,배민,default
7,10,비비큐 맵소디,default
8,11,푸라닭 깐풍치킨,default
9,12,교촌치킨 간장/레드 반반,default


In [11]:
# Users
show('SELECT * FROM user ORDER BY id')

,id,user_id,password
0,1,lbk,1234
1,2,kbl,1234


In [12]:
# Reviews (latest first)
show(
    'SELECT id, user_id, created_at, updated_at, content FROM review ORDER BY COALESCE(updated_at, created_at) DESC'
)

,id,user_id,created_at,updated_at,content
0,22,1,2025-12-29T09:21:47.577Z,2025-12-29T09:21:47.577Z,앱에서 맥주 재고 확인하고 직접 마트 가니까 아무리 찾아도 못찾겠다. 집에 와서 보...
1,21,2,2025-12-29T09:20:45.197Z,2025-12-29T09:20:45.197Z,4개 한셋트 치약 1+1 세일한다기에 사고 보니 3개 한셋트 치약이었다. 영수증엔 ...
2,20,2,2025-12-29T09:20:17.511Z,2025-12-29T09:20:17.511Z,홈플러스 앱으로 와인 주문: 10만원 이상 사면 20%할인한다기에 가격 맞춰 3병 ...
3,19,1,2025-12-29T09:19:50.904Z,2025-12-29T09:19:50.904Z,정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어...
4,18,1,2025-12-29T09:18:58.976Z,2025-12-29T09:18:58.976Z,예전에 땡겨요에서 세일하길래 무슨 요리 배달 주문했더니 배달에 한시간 이상 걸린다고...
5,17,2,2025-12-29T09:18:02.078Z,2025-12-29T09:18:02.078Z,며칠이 니자도 발뒤꿈치가 아파 미금역 근처 정형외과를 찾아갔다. 아킬레스건염 같은 ...
6,16,2,2025-12-29T09:17:13.163Z,2025-12-29T09:17:13.163Z,발뒤꿈치가 아파 정형외과를 찾는데 저번에 비싼 주사 맞은데를 피해서 다른 병원을 갔...
7,15,2,2025-12-29T09:16:18.396Z,2025-12-29T09:16:18.396Z,엄지 발가락이 아파서 동네 정형외과 갔다가 얼떨결에 비급여 10만원짜리 주사 맞고 ...
8,14,1,2025-12-29T09:15:40.337Z,2025-12-29T09:15:40.337Z,속초에서 시장 구경하고 돌아가는 길 닭강정 사는걸 깜빡했다. 많은 사람들이 만석닭강...
9,13,1,2025-12-29T09:15:16.784Z,2025-12-29T09:15:16.784Z,마트에서 풀무원 감자전/오징어부추전 구입. 에어프라이로 조리했는데 기름 범벅. 후라...


In [24]:
# Entities
df = show('SELECT * FROM entity ORDER BY id')
df['name'].duplicated()

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
Name: name, dtype: bool

In [28]:
df = df.loc[~df['id'].isin([4, 5])]
df = df.set_index('id')
df

,name,type,level,parent_id
id,,,,
1,BHC 쏘마치,None,None,None
2,처가집 양념통닭 K마라치킨,None,None,None
3,멕시카나,None,None,None
6,노랑통닭,None,None,None
9,배민,None,None,None
10,비비큐 맵소디,None,None,None
11,푸라닭 깐풍치킨,None,None,None
12,교촌치킨 간장/레드 반반,None,None,None
13,자담치킨,None,None,None


In [36]:
#ids = [1,2,10,11,12,19,26,27]
#df.loc[ids, 'type'] = 'product'

#ids = [3,6,13,20,21,23,25,28,29,30,32,34,35,36]
ids = [9,14,22]
df.loc[ids, 'type'] = 'vendor'
#df.loc[df['type'].isna()]

In [38]:
ids = [24,31]
df.loc[ids, 'type'] = 'location'
df.loc[df['type'].isna()]

,name,type,level,parent_id
id,,,,


In [43]:
df

,name,type,level,parent_id
id,,,,
1,BHC 쏘마치,product,None,None
2,처가집 양념통닭 K마라치킨,product,None,None
3,멕시카나,vendor,None,None
6,노랑통닭,vendor,None,None
9,배민,vendor,None,None
10,비비큐 맵소디,product,None,None
11,푸라닭 깐풍치킨,product,None,None
12,교촌치킨 간장/레드 반반,product,None,None
13,자담치킨,vendor,None,None


In [46]:
idx = [37, 38, 39]

df = df.reindex(df.index.union(idx))

df.loc[idx, ['name', 'type']] = [
    ['치킨', 'product'],
    ['병원', 'service'],
    ['마트', 'service'],
]

In [48]:
df.to_csv('entities.csv')

In [14]:
# Review-entity links
show(
    'SELECT review_id, entity_id FROM review_entity ORDER BY review_id, entity_id'
)

,review_id,entity_id
0,1,1
1,2,2
2,3,3
3,3,9
4,4,10
5,5,6
6,6,11
7,7,12
8,8,13
9,8,14


In [19]:
# Joined view: reviews with entity names
show(
    '''
    SELECT r.id, r.user_id, r.created_at, r.updated_at, r.content,
           GROUP_CONCAT(e.name, ', ') AS entities
    FROM review r
    LEFT JOIN review_entity re ON r.id = re.review_id
    LEFT JOIN entity e ON e.id = re.entity_id
    GROUP BY r.id
    ORDER BY COALESCE(r.updated_at, r.created_at) DESC
    '''
)

,id,user_id,created_at,updated_at,content,entities
0,22,1,2025-12-29T09:21:47.577Z,2025-12-29T09:21:47.577Z,앱에서 맥주 재고 확인하고 직접 마트 가니까 아무리 찾아도 못찾겠다. 집에 와서 보...,이마트
1,21,2,2025-12-29T09:20:45.197Z,2025-12-29T09:20:45.197Z,4개 한셋트 치약 1+1 세일한다기에 사고 보니 3개 한셋트 치약이었다. 영수증엔 ...,이마트
2,20,2,2025-12-29T09:20:17.511Z,2025-12-29T09:20:17.511Z,홈플러스 앱으로 와인 주문: 10만원 이상 사면 20%할인한다기에 가격 맞춰 3병 ...,홈플러스
3,19,1,2025-12-29T09:19:50.904Z,2025-12-29T09:19:50.904Z,정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어...,정동마트
4,18,1,2025-12-29T09:18:58.976Z,2025-12-29T09:18:58.976Z,예전에 땡겨요에서 세일하길래 무슨 요리 배달 주문했더니 배달에 한시간 이상 걸린다고...,땡겨요
5,17,2,2025-12-29T09:18:02.078Z,2025-12-29T09:18:02.078Z,며칠이 니자도 발뒤꿈치가 아파 미금역 근처 정형외과를 찾아갔다. 아킬레스건염 같은 ...,미금더튼튼의원
6,16,2,2025-12-29T09:17:13.163Z,2025-12-29T09:17:13.163Z,발뒤꿈치가 아파 정형외과를 찾는데 저번에 비싼 주사 맞은데를 피해서 다른 병원을 갔...,"이주철마취과의원, 분당"
7,15,2,2025-12-29T09:16:18.396Z,2025-12-29T09:16:18.396Z,엄지 발가락이 아파서 동네 정형외과 갔다가 얼떨결에 비급여 10만원짜리 주사 맞고 ...,한사랑 정형외과의원
8,14,1,2025-12-29T09:15:40.337Z,2025-12-29T09:15:40.337Z,속초에서 시장 구경하고 돌아가는 길 닭강정 사는걸 깜빡했다. 많은 사람들이 만석닭강...,속초 동해닭강정
9,13,1,2025-12-29T09:15:16.784Z,2025-12-29T09:15:16.784Z,마트에서 풀무원 감자전/오징어부추전 구입. 에어프라이로 조리했는데 기름 범벅. 후라...,"풀무원 감자전, 풀무원 오징어부추전"


In [16]:
conn.close()

# entity

In [1]:
import pandas as pd

In [9]:
df = pd.read_csv('entities.csv')
#df[['id','level','parent_id']] = df[['id','level','parent_id']].astype(int)
df = df.set_index('id')
df

,name,type,level,parent_id
id,,,,
1,BHC 쏘마치,product,NaN,NaN
2,처가집 양념통닭 K마라치킨,product,NaN,NaN
3,멕시카나,vendor,NaN,NaN
6,노랑통닭,vendor,NaN,NaN
9,배민,vendor,NaN,NaN
10,비비큐 맵소디,product,NaN,NaN
11,푸라닭 깐풍치킨,product,NaN,NaN
12,교촌치킨 간장/레드 반반,product,NaN,NaN
13,자담치킨,vendor,NaN,NaN


In [10]:
ids = [1,2,10,11,19]
df.loc[ids, 'parent_id'] = 37
#df.loc[ids]

In [17]:
id = df.loc[df['name']=='치킨'].index[0]
df.loc[df['parent_id']==id]

,name,type,level,parent_id
id,,,,
1,BHC 쏘마치,product,NaN,37.0
2,처가집 양념통닭 K마라치킨,product,NaN,37.0
10,비비큐 맵소디,product,NaN,37.0
11,푸라닭 깐풍치킨,product,NaN,37.0
19,고추바사삭,product,NaN,37.0
